In [11]:
import json
import os
import pandas as pd

print("=" * 70)
print("STEP 2: DATASET LOADING")
print("=" * 70)

# ── 1. Load Davidson ─────────────────────────────────────────
print("\n[1/4] Loading Davidson dataset...")

davidson = pd.read_csv("./data/raw/davidson_raw.csv")

# Davidson label system: 0 = hate speech, 1 = offensive, 2 = normal
# Remap to our system: 0 = normal, 1 = offensive, 2 = hate speech
davidson_label_map = {2: 0, 1: 1, 0: 2}

davidson_df = pd.DataFrame({
    "text"   : davidson["tweet"],
    "label"  : davidson["class"].map(davidson_label_map),
    "source" : "davidson"
})
print(f"   ✓ Loaded {len(davidson_df)} rows from Davidson")

# ── 2. Load Annotated Reddit Data ────────────────────────────
print("\n[2/4] Loading annotated Reddit data...")

annotated_folder = "./data/annotated/"
records = []

for filename in os.listdir(annotated_folder):
    if filename.endswith(".json"):
        with open(os.path.join(annotated_folder, filename), "r", encoding="utf-8") as f:
            data = json.load(f)
            records.extend(data)

reddit_raw = pd.DataFrame(records)

# Keep only annotated records
reddit_raw = reddit_raw[reddit_raw["annotation_status"] == "annotated"]

# Map string labels to numbers
reddit_label_map = {"normal": 0, "offensive": 1, "hate speech": 2}

reddit_df = pd.DataFrame({
    "text"   : reddit_raw["text"],
    "label"  : reddit_raw["label"].map(reddit_label_map),
    "source" : reddit_raw["platform"]  # or "reddit" if platform column doesn't exist
})
print(f"   ✓ Loaded {len(reddit_df)} rows from annotated data")

# ── 3. Load HackerNews Data (if exists) ──────────────────────
print("\n[3/4] Loading HackerNews data...")

hackernews_files = [
    "./data/raw/hackernews_20260521_165050.json",
    "./data/raw/hackernews_20260523_213137.json"
]

hackernews_records = []

for file_path in hackernews_files:
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list):
                hackernews_records.extend(data)
            else:
                hackernews_records.append(data)

if hackernews_records:
    df_hackernews = pd.DataFrame(hackernews_records)
    print(f"   ✓ Loaded {len(df_hackernews)} rows from HackerNews")
    
    # Check if HackerNews has labels (it may not)
    if "label" in df_hackernews.columns:
        # Already has labels
        hackernews_df = pd.DataFrame({
            "text": df_hackernews["text"],
            "label": df_hackernews["label"],
            "source": "hackernews"
        })
    else:
        # No labels - use for unlabeled analysis only
        print("   ⚠️ HackerNews data has no labels. Will be used for testing only.")
        hackernews_df = pd.DataFrame({
            "text": df_hackernews["text"],
            "label": -1,  # -1 indicates unlabeled
            "source": "hackernews"
        })
else:
    print("   ⚠️ No HackerNews files found. Skipping.")
    hackernews_df = pd.DataFrame()  # Empty dataframe

# ── 4. Combine All Data ──────────────────────────────────────
print("\n[4/4] Combining all data...")

# Start with Davidson and Reddit
df = pd.concat([davidson_df, reddit_df], ignore_index=True)

# Add HackerNews if it has data
if len(hackernews_df) > 0:
    df = pd.concat([df, hackernews_df], ignore_index=True)

# Add human-readable label column
label_names = {0: "normal", 1: "offensive", 2: "hate speech"}
df["label_name"] = df["label"].map(label_names)

# ── 5. Summary ───────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 2 COMPLETE - SUMMARY")
print("=" * 70)

print(f"\nDavidson       : {len(davidson_df)} rows")
print(f"Reddit         : {len(reddit_df)} rows")
if len(hackernews_df) > 0:
    print(f"HackerNews     : {len(hackernews_df)} rows")
print(f"Total combined : {len(df)} rows")

print(f"\nSources:")
print(df["source"].value_counts())

print(f"\nLabel distribution:")
print(df[df["label"] != -1]["label_name"].value_counts())  # exclude unlabeled

print(f"\nColumns: {df.columns.tolist()}")

print(f"\nFirst 3 rows:")
print(df.head(3))

# Save combined data for next steps
df.to_csv("./data/processed/combined_data.csv", index=False)
print("\n✅ Saved combined data to: ./data/processed/combined_data.csv")
print("\n➡️ Next: Step 3 - Data Inspection")

STEP 2: DATASET LOADING

[1/4] Loading Davidson dataset...
   ✓ Loaded 24783 rows from Davidson

[2/4] Loading annotated Reddit data...
   ✓ Loaded 3489 rows from annotated data

[3/4] Loading HackerNews data...
   ✓ Loaded 40 rows from HackerNews

[4/4] Combining all data...

STEP 2 COMPLETE - SUMMARY

Davidson       : 24783 rows
Reddit         : 3489 rows
HackerNews     : 40 rows
Total combined : 28312 rows

Sources:
source
davidson      24783
reddit         3449
hackernews       80
Name: count, dtype: int64

Label distribution:
label_name
offensive      19607
normal          7053
hate speech     1430
Name: count, dtype: int64

Columns: ['text', 'label', 'source', 'label_name']

First 3 rows:
                                                text label    source  \
0  !!! RT @mayasolovely: As a woman you shouldn't...   0.0  davidson   
1  !!!!! RT @mleew17: boy dats cold...tyga dwn ba...   1.0  davidson   
2  !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...   1.0  davidson   

  label

In [12]:
# ============================================================
# STEP 3 — DATA INSPECTION (shape, columns, dtypes)
# ============================================================

import pandas as pd
import numpy as np

# Load the combined data from Step 2
df = pd.read_csv("./data/processed/combined_data.csv")

print("=" * 70)
print("STEP 3: DATA INSPECTION")
print("=" * 70)

# ---------------------------------------------------------------------
# 3.1 Basic Information
# ---------------------------------------------------------------------
print("\n[3.1] BASIC INFORMATION")
print("-" * 50)
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ---------------------------------------------------------------------
# 3.2 Data Types
# ---------------------------------------------------------------------
print("\n[3.2] DATA TYPES")
print("-" * 50)
print(df.dtypes)

# ---------------------------------------------------------------------
# 3.3 Missing Values Analysis
# ---------------------------------------------------------------------
print("\n[3.3] MISSING VALUES ANALYSIS")
print("-" * 50)
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percent
})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if len(missing_df) > 0:
    print(missing_df)
else:
    print("✓ No missing values found in any column")

# ---------------------------------------------------------------------
# 3.4 Unique Values per Column
# ---------------------------------------------------------------------
print("\n[3.4] UNIQUE VALUES PER COLUMN")
print("-" * 50)
for col in df.columns:
    n_unique = df[col].nunique()
    print(f"{col:15} : {n_unique:,} unique values")

# ---------------------------------------------------------------------
# 3.5 Statistical Summary (Numerical Columns)
# ---------------------------------------------------------------------
print("\n[3.5] STATISTICAL SUMMARY (Numerical Columns)")
print("-" * 50)
print(df.describe())

# ---------------------------------------------------------------------
# 3.6 Text Column Inspection
# ---------------------------------------------------------------------
print("\n[3.6] TEXT COLUMN INSPECTION")
print("-" * 50)

# Check for empty strings
empty_texts = df[df['text'].astype(str).str.strip() == '']
print(f"Empty text entries: {len(empty_texts)}")

# Check for very short texts (less than 3 characters)
short_texts = df[df['text'].astype(str).str.len() < 3]
print(f"Very short texts (<3 chars): {len(short_texts)}")

# Sample of texts
print("\n📝 Sample texts from each label:")
for label in sorted(df['label'].unique()):
    label_name = {0: "normal", 1: "offensive", 2: "hate speech", -1: "unlabeled"}.get(label, "unknown")
    sample = df[df['label'] == label]['text'].iloc[0] if len(df[df['label'] == label]) > 0 else "No data"
    print(f"\n  [{label_name}]: {sample[:100]}...")

# ---------------------------------------------------------------------
# 3.7 Label Column Detailed Analysis
# ---------------------------------------------------------------------
print("\n[3.7] LABEL COLUMN ANALYSIS")
print("-" * 50)

label_counts = df['label'].value_counts().sort_index()
label_names = {0: "normal", 1: "offensive", 2: "hate speech", -1: "unlabeled"}

for label, count in label_counts.items():
    name = label_names.get(label, "unknown")
    percent = (count / len(df)) * 100
    print(f"  {name:12} (label={label}): {count:6,} rows ({percent:.2f}%)")

# ---------------------------------------------------------------------
# 3.8 Source Column Analysis
# ---------------------------------------------------------------------
print("\n[3.8] SOURCE COLUMN ANALYSIS")
print("-" * 50)
source_counts = df['source'].value_counts()
for source, count in source_counts.items():
    percent = (count / len(df)) * 100
    print(f"  {source:12}: {count:6,} rows ({percent:.2f}%)")

# ---------------------------------------------------------------------
# 3.9 Cross-tabulation: Source vs Label
# ---------------------------------------------------------------------
print("\n[3.9] SOURCE vs LABEL (Cross-tabulation)")
print("-" * 50)
cross_tab = pd.crosstab(df['source'], df['label'])
print(cross_tab)

# ---------------------------------------------------------------------
# 3.10 Save Inspection Report
# ---------------------------------------------------------------------
print("\n[3.10] SAVING INSPECTION REPORT")
print("-" * 50)

# Save missing values report
missing_df.to_csv("./data/processed/inspection_missing_values.csv")
print("✓ Saved: ./data/processed/inspection_missing_values.csv")

# Save cross-tabulation
cross_tab.to_csv("./data/processed/inspection_source_label.csv")
print("✓ Saved: ./data/processed/inspection_source_label.csv")

# ---------------------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 3 COMPLETE - SUMMARY")
print("=" * 70)

print(f"""
Key findings:
- Total rows: {len(df):,}
- Total columns: {len(df.columns)}
- Missing values: {df.isnull().sum().sum()} total ({df.isnull().sum().sum()/len(df)*100:.2f}% of all cells)
- Unique labels: {df['label'].nunique()}
- Unique sources: {df['source'].nunique()}
- Empty texts: {len(empty_texts)}
- Very short texts: {len(short_texts)}
""")

print("\n✅ Step 3 complete. Ready for Step 4: Data Cleaning")

STEP 3: DATA INSPECTION

[3.1] BASIC INFORMATION
--------------------------------------------------
Shape: 28312 rows, 4 columns
Memory usage: 8.07 MB

[3.2] DATA TYPES
--------------------------------------------------
text              str
label         float64
source            str
label_name        str
dtype: object

[3.3] MISSING VALUES ANALYSIS
--------------------------------------------------
            Missing Count  Percentage (%)
label                 222         0.78412
label_name            222         0.78412

[3.4] UNIQUE VALUES PER COLUMN
--------------------------------------------------
text            : 28,215 unique values
label           : 3 unique values
source          : 3 unique values
label_name      : 3 unique values

[3.5] STATISTICAL SUMMARY (Numerical Columns)
--------------------------------------------------
              label
count  28090.000000
mean       0.799822
std        0.511793
min        0.000000
25%        0.000000
50%        1.000000
75%     